In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'IND': ['Jarace Walker', 'Kobe Brown', 'Ben Sheppard'], 'HOU': ['Tari Eason'], 'GSW': ['Gui Santos']}

Out Players:
{'CHI': ['Anfernee Simons', 'Josh Giddey', 'Matas Buzelis', 'Nick Richards', 'Isaac Okoro'], 'WAS': ['Anthony Davis', 'Tre Johnson', 'Trae Young', 'Bilal Coulibaly', "D'Angelo Russell", 'Alex Sarr', 'Jaden Hardy', 'Tristan Vukcevic'], 'MIA': ['Dru Smith', 'Nikola Jović', 'Terry Rozier'], 'TOR': ['Trayce Jackson-Davis', 'Chucky Hepburn'], 'IND': ['T.J. McConnell', 'Aaron Nesmith', 'Andrew Nembhard', 'Pascal Siakam'], 'BKN': ['Ziaire Williams', 'Noah Clowney', 'Josh Minott', 'Nolan Traore', 'Nic Claxton', 'Terance Mann'], 'BOS': ['Jaylen Brown'], 'NYK': ['Tyler Kolek'], 'PHI': ['Johni Broome', 'Joel Embiid', 'Cameron Payne'], 'HOU': ['Fred VanVleet'], 'LAL': ['Jaxson Hayes', 'Austin Reaves', 'Marcus Smart', 'Luka Dončić'], 'GSW': ['Al Horford', 'Will Richard', 'Stephen Curry', 'Quinten Post', 'Kristaps Porziņģis']}
Note: IND (Pacers) has 3 confirmed 

### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
103,NaN,2025-26,1642349,Ajay Mitchell,Ajay,1610612760,OKC,Oklahoma City Thunder,22501163,2026-04-08T00:00:00,OKC @ LAC,W,24.233333,3,6,0.500,0,1,0.00,1,2,0.5,0,1,1,3,0,1,0,0,1,1,7,14,15.7,0,0,13.0,1,24:14,1,129.3,130.0,130.0,101.9,100.0,100.0,27.4,30.0,30.0,0.143,0.0,30.0,0.00,0.037,0.021,0.0,0.0,0.500,0.509,0.125,0.127,99.35,100.03,83.36,100.03,0.065,50,3.0,6.0,48,83,0.578,13,34,0.382,19,24,0.792,6,38,44,30,13.0,9,7,3,23,18,128,18.0,127.3,129.3,110.5,111.1,16.8,18.2,0.625,2.31,21.7,0.237,0.755,0.538,0.131,0.657,0.684,100.1,99.0,82.50,99,0.592,1610612746,LAC,LA Clippers,40,86,0.465,14,32,0.438,16,24,0.667,10,26,36,27,13.0,9,3,7,18,23,110,-18.0,110.5,111.1,127.3,129.3,-16.8,-18.2,0.675,2.08,19.4,0.245,0.763,0.462,0.131,0.547,0.570,100.1,99.0,82.50,99,0.408,NaN,SG,23.0
102,NaN,2025-26,1630577,Julian Champagnie,Julian,1610612759,SAS,San Antonio Spurs,22501161,2026-04-08T00:00:00,SAS vs. POR,W,26.833333,1,6,0.167,0,3,0.00,0,0,0.0,0,4,4,3,0,0,2,0,1,1,2,6,17.3,0,0,13.0,1,26:50,1,118.7,118.6,118.6,109.6,110.3,110.3,9.1,8.3,8.3,0.111,0.0,33.3,0.00,0.154,0.075,0.0,0.0,0.167,0.167,0.088,0.091,104.97,104.65,87.20,104.65,0.031,59,1.0,6.0,43,88,0.489,11,29,0.379,15,19,0.789,11,34,45,28,17.0,13,6,5,14,15,112,11.0,109.4,110.9,98.6,100.0,10.8,10.9,0.651,1.65,19.6,0.306,0.712,0.515,0.168,0.551,0.581,102.4,101.0,84.17,101,0.565,1610612757,POR,Portland Trail Blazers,42,93,0.452,12,37,0.324,5,10,0.500,11,32,43,26,16.0,9,5,6,15,14,101,-11.0,98.6,100.0,109.4,110.9,-10.8,-10.9,0.619,1.63,18.4,0.288,0.694,0.485,0.158,0.516,0.518,102.4,101.0,84.17,101,0.435,F,SF,24.0
101,NaN,2025-26,1629622,Max Strus,Max,1610612739,CLE,Cleveland Cavaliers,22501158,2026-04-08T00:00:00,CLE vs. ATL,W,21.233333,3,7,0.429,2,5,0.40,0,0,0.0,0,2,2,2,1,0,0,1,1,2,8,-2,12.4,0,0,14.0,1,21:14,1,109.2,110.9,110.9,112.9,110.4,110.4,-3.7,0.5,0.5,0.133,2.0,20.0,0.00,0.111,0.048,10.0,10.0,0.571,0.571,0.154,0.158,105.89,106.25,88.54,106.25,0.071,46,3.0,7.0,41,88,0.466,12,33,0.364,28,35,0.800,10,37,47,22,11.0,6,6,6,14,25,122,6.0,116.9,117.3,107.8,110.5,9.1,6.8,0.537,2.00,15.9,0.240,0.717,0.485,0.106,0.534,0.590,106.0,104.5,87.08,104,0.552,1610612737,ATL,Atlanta Hawks,47,98,0.480,12,34,0.353,10,15,0.667,10,33,43,23,13.0,8,6,6,25,14,116,-6.0,107.8,110.5,116.9,117.3,-9.1,-6.8,0.489,1.77,16.1,0.283,0.760,0.515,0.124,0.541,0.554,106.0,104.5,87.08,105,0.448,NaN,SF,29.0
79,NaN,2025-26,1642942,Jahmai Mashack,J

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260409_151832.json


,home_team,away_team,commence_time,bookmakers
0,Washington Wizards,Chicago Bulls,2026-04-09 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Toronto Raptors,Miami Heat,2026-04-09 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,New York Knicks,Boston Celtics,2026-04-09 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,Brooklyn Nets,Indiana Pacers,2026-04-09 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,Houston Rockets,Philadelphia 76ers,2026-04-10 00:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-09 15:18:33
US latest pull: 2026-04-09 15:01:16


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Betr DFS,player_points,Collin Sexton,Over,22.5,-137,2026-04-09,2026-04-09T22:17:41Z,2026-04-09 15:18:33
1,Betr DFS,player_points,Collin Sexton,Under,22.5,-137,2026-04-09,2026-04-09T22:17:41Z,2026-04-09 15:18:33
2,Betr DFS,player_points,Tre Jones,Over,18.5,-137,2026-04-09,2026-04-09T22:17:41Z,2026-04-09 15:18:33
3,Betr DFS,player_points,Tre Jones,Under,18.5,-137,2026-04-09,2026-04-09T22:17:41Z,2026-04-09 15:18:33
4,Betr DFS,player_points,Leonard Miller,Over,16.5,-137,2026-04-09,2026-04-09T22:17:41Z,2026-04-09 15:18:33


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Leonard Miller: single positional indexer is out-of-bounds
[SKIP] Anthony Gill: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] Kobe Brown: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] Leonard Miller: single positional indexer is out-of-bounds
[SKIP] Kobe Brown: single positional indexer is out-of-bounds


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Will Riley,AST,21.36,31.78,37.47,0.0319,0.0991,0.1674,0.68,3.15,6.27,"[0.0, 0.1024590163934426, 0.1298026998961578, ..."
1,Davion Mitchell,AST,16.29,25.88,34.28,0.1015,0.1842,0.2967,1.65,4.77,10.17,"[0.1647446457990115, 0.2074688796680497, 0.187..."
2,Immanuel Quickley,AST,14.32,25.52,35.19,0.0716,0.1722,0.2837,1.03,4.39,9.98,"[0.2910737386804657, 0.1227747084100675, 0.0, ..."
3,Scottie Barnes,AST,10.54,27.34,37.10,0.1196,0.2227,0.3574,1.26,6.09,13.26,"[0.1125809175344779, 0.1271860095389507, 0.212..."
4,Brandon Ingram,AST,6.75,27.20,37.02,0.0444,0.1369,0.2579,0.30,3.72,9.55,"[0.0670690811535882, 0.1913875598086124, 0.083..."
5,Jamal Shead,AST,16.16,21.84,27.82,0.1367,0.2247,0.3796,2.21,4.91,10.56,"[0.1483679525222552, 0.3153330705557745, 0.122..."
6,Jarace Walker,AST,14.54,21.53,29.95,0.0435,0.1214,0.2072,0.63,2.61,6.20,"[0.0799041150619256, 0.163025758069775, 0.1862..."
7,Tyrese Maxey,AST,8.47,30.55,42.11,0.1014,0.1688,0.2761,0.86,5.16,11.63,"[0.077359463641052, 0.1932100469224399, 0.2256..."
8,Reed Sheppard,AST,18.52,28.08,35.49,0.0518,0.1344,0.2473,0.96,3.77,8.78,"[0.07380073800738, 0.0935745477230193, 0.09705..."
9,Draymond Green,AST,12.80,28.19,36.09,0.1494,0.2284,0.3246,1.91,6.44,11.71,"[0.1532332209623046, 0.3387742531567601, 0.135..."


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER
46,Amen Thompson,REB,8.0,31.01,7.03,0.320,0.598
21,Ja'Kobe Walter,AST,1.5,23.41,1.40,0.472,0.528
43,Alperen Sengun,REB,8.5,28.73,9.35,0.380,0.621
116,Kevin Durant,PTS,25.5,29.52,22.15,0.299,0.701
89,Brandon Ingram,PTS,20.5,27.20,17.57,0.289,0.711
109,Quentin Grimes,PTS,9.5,24.84,10.86,0.583,0.417
61,Quentin Grimes,REB,2.5,24.84,3.06,0.657,0.343
41,Andre Drummond,REB,8.5,23.58,10.17,0.621,0.379
92,Collin Murray-Boyles,PTS,10.5,19.89,8.18,0.465,0.535
38,Josh Hart,REB,7.5,27.33,6.73,0.397,0.603


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
115,Tari Eason,PTS,9.5,24.96,11.45,0.594,0.406,PTS,Underdog,Philadelphia 76ers,-6.0,226.5,114.9,17.0,100.28,16.0,-120.0,105.0,0.545,0.488,9.7,9.5,6.07,0.2,0.0,-0.033,0.513,0.487,-5.95,-0.17,0.6,0.5,0.47,0.60,23.29,5.39,0.18,0.04,16.00,3.0
1,Davion Mitchell,AST,5.5,25.88,4.77,0.441,0.559,AST,Underdog,Toronto Raptors,3.5,236.5,112.0,5.0,99.36,21.0,-104.0,105.0,0.510,0.488,5.7,6.0,2.00,0.2,0.5,-0.100,0.540,0.460,5.92,-5.70,0.8,0.6,0.40,0.49,30.22,6.30,0.15,0.04,3.25,4.0
89,Brandon Ingram,PTS,20.5,27.20,17.57,0.289,0.711,PTS,Underdog,Miami Heat,-3.5,236.5,113.4,11.0,104.37,1.0,102.0,-120.0,0.495,0.545,18.5,18.0,7.26,-3.0,-3.5,0.413,0.340,0.660,-31.32,21.00,0.4,0.3,0.33,0.49,31.64,4.12,0.24,0.04,21.00,3.0
54,Davion Mitchell,REB,2.5,25.88,2.34,0.472,0.528,REB,Underdog,Toronto Raptors,3.5,236.5,112.0,5.0,99.36,21.0,100.0,-112.0,0.500,0.528,2.6,2.5,1.17,0.1,0.0,-0.085,0.534,0.466,6.80,-11.79,0.6,0.5,0.53,0.46,30.22,6.30,0.15,0.04,2.00,4.0
77,Guerschon Yabusele,PTS,12.5,28.35,10.99,0.562,0.438,PTS,Underdog,Washington Wizards,-6.5,248.5,121.3,29.0,102.46,6.0,-120.0,-108.0,0.545,0.519,10.2,9.5,4.78,-2.3,-3.0,0.481,0.315,0.685,-42.25,31.93,0.4,0.3,0.33,0.24,24.45,5.52,0.16,0.04,13.00,4.0


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
62,Deandre Ayton,REB,8.5,25.86,9.06,0.417,0.583,REB,PrizePicks,Golden State Warriors,2.5,221.5,114.0,16.0,100.19,17.0,-137.0,-137.0,0.578,0.578,6.1,5.5,3.48,-1.9,-2.5,0.546,0.293,0.707,-49.31,22.31,0.2,0.3,0.47,0.50,25.31,4.72,0.15,0.04,9.00,3.0
125,Draymond Green,PTS,9.5,28.19,8.16,0.361,0.639,PTS,PrizePicks,Los Angeles Lakers,-2.5,221.5,116.1,20.0,99.34,22.0,-137.0,-137.0,0.578,0.578,8.0,7.5,4.88,-1.0,-1.5,0.205,0.419,0.581,-27.52,0.51,0.4,0.4,0.47,0.41,29.92,5.81,0.14,0.05,8.00,6.0
37,Sandro Mamukelashvili,REB,4.5,21.08,5.62,0.605,0.395,REB,PrizePicks,Miami Heat,-3.5,236.5,113.4,11.0,104.37,1.0,-113.0,-105.0,0.531,0.512,5.6,6.0,1.71,1.1,1.5,-0.643,0.740,0.260,39.49,-49.24,1.0,0.7,0.67,0.40,23.02,5.67,0.21,0.06,4.20,5.0
116,Kevin Durant,PTS,25.5,29.52,22.15,0.299,0.701,PTS,PrizePicks,Philadelphia 76ers,-6.0,226.5,114.9,17.0,100.28,16.0,-137.0,-137.0,0.578,0.578,26.8,26.0,6.00,1.8,1.0,-0.300,0.618,0.382,6.91,-33.92,0.4,0.5,0.40,0.51,36.03,4.28,0.27,0.04,31.33,3.0
127,Gui Santos,PTS,14.5,24.28,12.06,0.448,0.552,PTS,PrizePicks,Los Angeles Lakers,-2.5,221.5,116.1,20.0,99.34,22.0,-118.0,-110.0,0.541,0.524,16.9,15.5,8.74,2.4,1.0,-0.275,0.608,0.392,12.33,-25.16,0.8,0.6,0.73,0.19,30.87,3.57,0.22,0.06,7.00,5.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
113,Alperen Sengun,PTS,19.5,28.73,17.68,0.387,0.613,PTS,Betr DFS,Philadelphia 76ers,-6.0,226.5,114.9,17.0,100.28,16.0,100.0,-113.0,0.500,0.531,22.5,21.5,8.53,3.0,2.0,-0.352,0.638,0.362,27.60,-31.76,0.4,0.5,0.40,0.47,32.94,4.82,0.24,0.05,16.00,3.0
108,Ben Saraf,PTS,15.5,28.25,9.85,0.270,0.730,PTS,Betr DFS,Indiana Pacers,3.0,224.5,118.3,26.0,101.68,8.0,-114.0,-114.0,0.533,0.533,11.2,10.0,6.27,-4.3,-5.5,0.686,0.246,0.754,-53.82,41.54,0.2,0.2,0.13,0.05,25.74,5.43,0.25,0.06,12.00,1.0
2,Immanuel Quickley,AST,5.0,25.52,4.39,0.395,0.495,AST,Betr DFS,Miami Heat,-3.5,236.5,113.4,11.0,104.37,1.0,-137.0,-137.0,0.578,0.578,5.2,6.5,2.74,0.2,1.5,-0.073,0.529,0.471,-8.49,-18.52,0.6,0.6,0.60,0.55,30.33,5.86,0.18,0.04,4.50,4.0
5,Jamal Shead,AST,5.5,21.84,4.91,0.571,0.429,AST,Betr DFS,Miami Heat,-3.5,236.5,113.4,11.0,104.37,1.0,102.0,-110.0,0.495,0.524,8.3,7.5,3.30,2.8,2.0,-0.848,0.802,0.198,62.00,-62.20,0.8,0.8,0.60,0.33,26.82,4.81,0.13,0.03,4.86,7.0
50,Brandin Podziemski,REB,5.5,29.25,5.73,0.486,0.514,REB,Betr DFS,Los Angeles Lakers,-2.5,221.5,116.1,20.0,99.34,22.0,115.0,-149.0,0.465,0.598,5.0,5.5,3.40,-0.5,0.0,0.147,0.442,0.558,-4.97,-6.75,0.2,0.5,0.47,0.42,31.42,6.66,0.21,0.04,6.00,7.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
50,Brandin Podziemski,REB,5.5,29.25,5.73,0.486,0.514,REB,DraftKings Pick6,Los Angeles Lakers,-2.5,221.5,116.1,20.0,99.34,22.0,115.0,-149.0,0.465,0.598,5.0,5.5,3.40,-0.5,0.0,0.147,0.442,0.558,-4.97,-6.75,0.2,0.5,0.47,0.42,31.42,6.66,0.21,0.04,6.00,7.0
27,Amen Thompson,AST,5.5,31.01,4.41,0.368,0.632,AST,DraftKings Pick6,Philadelphia 76ers,-6.0,226.5,114.9,17.0,100.28,16.0,112.0,-136.0,0.472,0.576,5.6,5.5,2.72,0.1,0.0,-0.037,0.515,0.485,9.18,-15.84,0.8,0.5,0.40,0.33,38.25,3.53,0.18,0.04,5.50,2.0
45,Kevin Durant,REB,5.5,29.52,4.93,0.315,0.685,REB,DraftKings Pick6,Philadelphia 76ers,-6.0,226.5,114.9,17.0,100.28,16.0,-108.0,-105.0,0.519,0.512,5.1,5.0,1.66,-0.4,-0.5,0.241,0.405,0.595,-22.00,16.17,0.4,0.4,0.40,0.49,36.03,4.28,0.27,0.04,5.67,3.0
3,Scottie Barnes,AST,6.5,27.34,6.09,0.519,0.481,AST,DraftKings Pick6,Miami Heat,-3.5,236.5,113.4,11.0,104.37,1.0,102.0,-120.0,0.495,0.545,9.6,10.0,3.27,3.1,3.5,-0.948,0.828,0.172,67.26,-68.47,0.6,0.7,0.60,0.35,29.68,4.12,0.22,0.03,6.17,6.0
62,Deandre Ayton,REB,8.5,25.86,9.06,0.417,0.583,REB,DraftKings Pick6,Golden State Warriors,2.5,221.5,114.0,16.0,100.19,17.0,105.0,-120.0,0.488,0.545,6.1,5.5,3.48,-2.4,-3.0,0.690,0.245,0.755,-49.78,38.42,0.2,0.3,0.47,0.50,25.31,4.72,0.15,0.04,9.00,3.0


In [13]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
121,LeBron James,PTS,25.5,29.57,21.07,0.168,0.832,PTS,DraftKings Pick6,Golden State Warriors,2.5,221.5,114.0,16.0,100.19,17.0,-115.0,-103.0,0.535,0.507,18.8,16.5,7.04,-6.7,-9.0,0.952,0.171,0.829,-68.03,63.39,0.2,0.2,0.13,0.35,34.54,4.02,0.22,0.04,28.83,6.0
37,Sandro Mamukelashvili,REB,4.5,21.08,5.62,0.605,0.395,REB,DraftKings Pick6,Miami Heat,-3.5,236.5,113.4,11.0,104.37,1.0,-113.0,-105.0,0.531,0.512,5.6,6.0,1.71,1.1,1.5,-0.643,0.740,0.260,39.49,-49.24,1.0,0.7,0.67,0.40,23.02,5.67,0.21,0.06,4.20,5.0
109,Quentin Grimes,PTS,9.5,24.84,10.86,0.583,0.417,PTS,PrizePicks,Houston Rockets,6.0,226.5,112.2,7.0,96.81,29.0,-102.0,-110.0,0.505,0.524,12.3,11.5,8.22,2.8,2.0,-0.341,0.633,0.367,25.36,-29.94,0.6,0.7,0.80,0.67,27.29,5.43,0.17,0.06,16.75,4.0
44,Tari Eason,REB,6.5,24.96,6.86,0.542,0.458,REB,Betr DFS,Philadelphia 76ers,-6.0,226.5,114.9,17.0,100.28,16.0,-137.0,-137.0,0.578,0.578,5.7,7.0,2.41,-0.3,1.0,0.124,0.451,0.549,-21.98,-5.03,0.6,0.6,0.67,0.42,23.29,5.39,0.18,0.04,8.00,3.0
96,Sam Hauser,PTS,9.5,22.56,8.44,0.384,0.616,PTS,DraftKings Pick6,New York Knicks,5.0,212.5,112.3,8.0,97.95,25.0,-103.0,-108.0,0.507,0.519,8.4,7.5,6.36,-1.1,-2.0,0.173,0.431,0.569,-15.06,9.59,0.4,0.3,0.33,0.38,24.97,5.49,0.12,0.04,8.67,6.0


### Get top EVs

In [14]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 84  |  Pairs: 173  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [15]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 38  |  Pairs: 63  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 76  |  Pairs: 186  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 74  |  Pairs: 210  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
